In [16]:
import os
import pandas as pd
import librosa
import json
from pydub import AudioSegment
from sklearn.model_selection import train_test_split
import shutil
import numpy as np
import tensorflow_hub as hub
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout


DATASET_DIR = r"C:\Users\Dell\Documents\Project\Listen_Up\Dataset"

In [140]:
flattened_dir = os.path.join(DATASET_DIR, "all_audio_files")
os.makedirs(flattened_dir, exist_ok=True)

for root, dirs, files in os.walk(DATASET_DIR):
    for file in files:
        if file.endswith((".wav", ".mp3")):
            src = os.path.join(root, file)
            dst = os.path.join(flattened_dir, file)
            if not os.path.exists(dst):  # Avoid overwriting
                shutil.copy(src, dst)

print(f"All files copied to {flattened_dir}")


All files copied to C:\Users\Dell\Documents\Project\Listen_Up\Dataset\all_audio_files


In [143]:
file_list_path = os.path.join(DATASET_DIR, "file_list.txt")

with open(file_list_path, 'w') as f:
    for filename in os.listdir(flattened_dir):
        if filename.endswith((".wav", ".mp3")):
            f.write(filename + "\n")

print(f"File list written to {file_list_path}")

File list written to C:\Users\Dell\Documents\Project\Listen_Up\Dataset\file_list.txt


In [331]:
class_folders = [d for d in os.listdir(DATASET_DIR) 
                 if os.path.isdir(os.path.join(DATASET_DIR, d)) and d != "all_audio_files"]

class_id_map = {name: idx for idx, name in enumerate(sorted(class_folders))}

indoor_keywords = [
    'Alarm', 'Appliance', 'Baby', 'Bathtub', 'Blender', 'Blinds', 'Book', 'Broom', 'Can', 'Celinig', 'Chair',
    'Child', 'Closet', 'Coffee', 'Computer', 'Cooking', 'Coughing', 'Cup', 'Curtain', 'Dishwasher', 'Door',
    'Drawer', 'Dryer', 'Electric', 'Fan', 'Faucet', 'Fireplace', 'Footsteps', 'Frying', 'Garbage', 'Hairdryer',
    'Hand', 'Heater', 'Iron', 'Kettle', 'Knocking', 'Laughter', 'Light', 'Oven', 'Remote', 'Spoon', 'Telephone',
    'Toaster', 'Toilet', 'Vacuum', 'Water'
]

environment_map = {}
for class_name in class_folders:
    if any(keyword.lower() in class_name.lower() for keyword in indoor_keywords):
        environment_map[class_name] = 'indoor'
    else:
        environment_map[class_name] = 'outdoor'

# Prepare labeled dataset
data = []

for class_name in class_folders:
    class_dir = os.path.join(DATASET_DIR, class_name)
    
    if os.path.exists(class_dir):
        for filename in os.listdir(class_dir):
            if filename.endswith((".wav", ".mp3")):
                file_path = os.path.join(class_dir, filename)
                class_id = class_id_map[class_name]
                environment = environment_map.get(class_name, "unknown")
                data.append({
                    "filename": filename,
                    "class_id": class_id,
                    "class_name": class_name,
                    "environment": environment
                })
    else:
        print(f"Warning: Folder '{class_name}' not found.")

df = pd.DataFrame(data)

# Save to CSV
csv_path = os.path.join(DATASET_DIR, "final_audioset.csv")
df.to_csv(csv_path, index=False)

print(f"CSV written to {csv_path}")


CSV written to C:\Users\Dell\Documents\Project\Listen_Up\Dataset\final_audioset.csv


In [150]:
SAMPLING_RATE = 16000
MAX_AUDIO_LENGTH = SAMPLING_RATE

Pre-Processing

In [153]:
def preprocess_audio(file_path):
    audio, sr = librosa.load(file_path, sr=SAMPLING_RATE, mono=True)

    # Fix audio length to 1 second
    audio = librosa.util.fix_length(audio, size=MAX_AUDIO_LENGTH)

    # Compute Mel-spectrogram
    mel_spectrogram = librosa.feature.melspectrogram(
        y=audio,
        sr=SAMPLING_RATE,
        n_mels=64,
        n_fft=1024,
        hop_length=512
    )
    mel_spectrogram = librosa.power_to_db(mel_spectrogram, ref=np.max)
    mel_spectrogram = np.expand_dims(mel_spectrogram, axis=-1)  # Add channel dimension
    return mel_spectrogram

In [283]:
dataset_path = r"C:\Users\Dell\Documents\Project\Listen_Up\Dataset"
audio_features = []
y_labels = []

for root, dirs, files in os.walk(dataset_path):
       for file in files:
        if file.lower().endswith((".wav", ".mp3")):
            file_path = os.path.join(root, file)
            try:
                features = preprocess_audio(file_path)
                audio_features.append(features)

                class_name = os.path.basename(root)
                y_labels.append(class_name)

                print(f"Processed {file_path} → Label: {class_name}")
            except Exception as e:
                print(f"Error processing {file_path}: {e}")

Processed C:\Users\Dell\Documents\Project\Listen_Up\Dataset\Alarm clock ringing\013153_xylophone-clock-57998.mp3 → Label: Alarm clock ringing
Processed C:\Users\Dell\Documents\Project\Listen_Up\Dataset\Alarm clock ringing\043487_ticking-clock-created-for-a-trailer-to-create-tension-73787.mp3 → Label: Alarm clock ringing
Processed C:\Users\Dell\Documents\Project\Listen_Up\Dataset\Alarm clock ringing\alarm-clock-70648.mp3 → Label: Alarm clock ringing
Processed C:\Users\Dell\Documents\Project\Listen_Up\Dataset\Alarm clock ringing\alarm-clock-90867.mp3 → Label: Alarm clock ringing
Processed C:\Users\Dell\Documents\Project\Listen_Up\Dataset\Alarm clock ringing\alarm-clock-short-6402.mp3 → Label: Alarm clock ringing
Processed C:\Users\Dell\Documents\Project\Listen_Up\Dataset\Alarm clock ringing\chiptune-alarm-clock-112869.mp3 → Label: Alarm clock ringing
Processed C:\Users\Dell\Documents\Project\Listen_Up\Dataset\Alarm clock ringing\clock-24340.mp3 → Label: Alarm clock ringing
Processed C:\U

In [285]:
print(f"\nNumber of features extracted: {len(audio_features)}")
print(f"First 5 labels: {y_labels[:5]}")


Number of features extracted: 2580
First 5 labels: ['Alarm clock ringing', 'Alarm clock ringing', 'Alarm clock ringing', 'Alarm clock ringing', 'Alarm clock ringing']


In [287]:
os.environ['TFHUB_CACHE_DIR'] = './tfhub_cache'
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')

print("Model Signature:", yamnet_model.signatures)

yamnet_input = tf.random.normal([16000])  # 16 kHz sample, 1-second length
yamnet_output = yamnet_model(yamnet_input)

print("Output of YAMNet:", yamnet_output)


Model Signature: _SignatureMap({'serving_default': <ConcreteFunction (*, waveform: TensorSpec(shape=(None,), dtype=tf.float32, name='waveform')) -> Dict[['output_0', TensorSpec(shape=(None, 521), dtype=tf.float32, name='output_0')], ['output_1', TensorSpec(shape=(None, 1024), dtype=tf.float32, name='output_1')], ['output_2', TensorSpec(shape=(None, 64), dtype=tf.float32, name='output_2')]] at 0x19A1C147650>})
Output of YAMNet: [<tf.Tensor: shape=(2, 521), dtype=float32, numpy=
array([[3.7459150e-02, 8.1035256e-04, 3.1447067e-04, ..., 8.5807679e-04,
        1.7958816e-02, 5.3610076e-04],
       [7.0962501e-03, 2.8395723e-04, 2.1963187e-05, ..., 1.8673887e-05,
        9.4740104e-04, 2.4546209e-06]], dtype=float32)>, <tf.Tensor: shape=(2, 1024), dtype=float32, numpy=
array([[0.        , 0.        , 0.        , ..., 0.02415923, 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.7518711 , 0.        ,
        0.        ]], dtype=float32)>, <tf.Tensor: shape=(

In [288]:
def extract_yamnet_features(file_path, sampling_rate=16000):
    # Load audio as mono, 16kHz
    audio, sr = librosa.load(file_path, sr=sampling_rate, mono=True)
    
    # Fix length to 1 sec (YAMNet expects at least 1 second)
    audio = librosa.util.fix_length(audio, size=sampling_rate)
    
    # Ensure 1D array (some libraries return 2D by accident)
    if len(audio.shape) > 1:
        audio = np.squeeze(audio)

    # Convert to tensor
    yamnet_input = tf.convert_to_tensor(audio, dtype=tf.float32)  # shape: (16000,)
    
    # Add batch dimension inside model call
    scores, embeddings, spectrogram = yamnet_model(yamnet_input)
    
    return embeddings.numpy()  # shape: (N, 1024)

In [291]:
class_folders = [d for d in os.listdir(DATASET_DIR) 
                 if os.path.isdir(os.path.join(dataset_dir, d)) and d != "all_audio_files"]
feature_list = []
label_list = []

In [293]:
valid_extensions = ('.wav', '.mp3')

for class_label, folder_name in enumerate(class_folders):
    class_folder_path = os.path.join(DATASET_DIR, folder_name)
    if os.path.isdir(class_folder_path):
        for audio_file in os.listdir(class_folder_path):
            audio_file_path = os.path.join(class_folder_path, audio_file)
            if audio_file_path.lower().endswith(valid_extensions):
                try:
                    embeddings = extract_yamnet_features(audio_file_path)
                    feature_list.append(embeddings.flatten())  # 1D feature vector
                    label_list.append(class_label)
                except Exception as e:
                    print(f"Error processing {audio_file_path}: {e}")

In [295]:
X = audio_features
y = y_labels

from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_categorical = to_categorical(y_encoded)

import numpy as np
X_array = np.array(X)

print("X shape:", X_array.shape)
print("y_categorical shape:", y_categorical.shape)

X shape: (2580, 64, 32)
y_categorical shape: (2580, 61)


In [297]:
model = Sequential()
model.add(Input(shape=(64, 32)))  
model.add(LSTM(128, return_sequences=False))
model.add(Dropout(0.3))
model.add(Dense(64, activation='relu'))
model.add(Dense(y_categorical.shape[1], activation='softmax'))

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm_4 (LSTM)                        │ (None, 128)                 │          82,432 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_7 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_10 (Dense)                     │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_11 (Dense)                     │ (None, 61)                  │           3,965 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 94,653 (369.74 KB)

 Trainable params: 94,653 (369.74 KB)

 Non-trainable params: 0 (0.00 B)

In [299]:
X_train, X_test, y_train, y_test = train_test_split(X_array, y_categorical, test_size=0.2, random_state=42)
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [301]:
history = model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.2)

Epoch 1/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 9s 73ms/step - accuracy: 0.3074 - loss: 3.5286 - val_accuracy: 0.4794 - val_loss: 2.8492
Epoch 2/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 3s 63ms/step - accuracy: 0.4770 - loss: 2.8431 - val_accuracy: 0.4794 - val_loss: 2.8218
Epoch 3/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 3s 62ms/step - accuracy: 0.5056 - loss: 2.7231 - val_accuracy: 0.4794 - val_loss: 2.8390
Epoch 4/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 3s 65ms/step - accuracy: 0.5077 - loss: 2.7196 - val_accuracy: 0.4794 - val_loss: 2.8287
Epoch 5/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 3s 62ms/step - accuracy: 0.4631 - loss: 2.8644 - val_accuracy: 0.4794 - val_loss: 2.8406
Epoch 6/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 3s 62ms/step - accuracy: 0.4997 - loss: 2.7153 - val_accuracy: 0.4794 - val_loss: 2.8417
Epoch 7/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 3s 62ms/step - accuracy: 0.4847 - loss: 2.7907 - val_accuracy: 0.4794 - val_loss: 2.8387
Epoch 8/20
52/52 ━━━━━━━━━━━━━━━━━━━━ 3s 65ms/step - accuracy: 0.4739 - loss: 2.8033 - val_accuracy: 0.4794 - v

In [303]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy:.2f}")

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.5058 - loss: 2.6367
Test Accuracy: 0.52


# Conducting Test Cases:

Test Case: Validate YAMNet Embedding Shape

In [266]:
def test_yamnet_embedding_shape():
    test_file = r"C:\Users\Dell\Downloads\Test case\test sounds\door_bell_campanello-porta_by-wikingo-voicemaker-95601.mp3"  # Replace with a real file path
    emb = extract_yamnet_features(test_file)
    assert len(emb.shape) == 2 and emb.shape[1] == 1024, f"Expected (N, 1024), got {emb.shape}"
    print("YAMNet embedding shape test passed.")


In [268]:
 test_yamnet_embedding_shape()

YAMNet embedding shape test passed.


Test Case: Label Encoding Check

In [271]:
def test_label_encoding():
    assert y_categorical.shape[1] == len(set(y_labels)), "Mismatch in number of classes and one-hot labels"
    print("Label encoding test passed.")


In [273]:
test_label_encoding()

Label encoding test passed.


Test Case: Model Prediction Shape

In [275]:
def test_lstm_model_prediction():
    sample_input = X_array[0].reshape(1, 64, 32)
    prediction = model.predict(sample_input)
    assert prediction.shape == (1, y_categorical.shape[1]), "LSTM model prediction shape is incorrect"
    print("LSTM  model prediction shape test passed.")


In [277]:
test_lstm_model_prediction()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 370ms/step
LSTM  model prediction shape test passed.


Model Prediction Test

In [281]:
from tensorflow.keras.models import load_model  
SAMPLING_RATE = 16000
REQUIRED_SHAPE = (64, 32)  # Expected by LSTM model

def preprocess_audio(file_path):
    audio, sr = librosa.load(file_path, sr=SAMPLING_RATE, mono=True)
    audio = librosa.util.fix_length(audio, size=SAMPLING_RATE)  # 1 second

    mel_spectrogram = librosa.feature.melspectrogram(
        y=audio,
        sr=SAMPLING_RATE,
        n_mels=REQUIRED_SHAPE[0],
        n_fft=1024,
        hop_length=512
    )
    mel_spectrogram_db = librosa.power_to_db(mel_spectrogram, ref=np.max)

    mel_resized = librosa.util.fix_length(mel_spectrogram_db, size=REQUIRED_SHAPE[1], axis=1)
    
    return mel_resized 

test_audio_path = r"C:\Users\Dell\Downloads\Test case\test sounds\2-glass-break-sounds-243249.mp3"
test_features = preprocess_audio(test_audio_path)

test_features = np.expand_dims(test_features, axis=0)  # shape: (1, 64, 32)

predictions = model.predict(test_features)
predicted_class_index = np.argmax(predictions)
predicted_label = label_encoder.inverse_transform([predicted_class_index])[0]

print(f"Predicted Class: {predicted_label}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
Predicted Class: Glass breaking
